In [1]:
# import standard python libraries
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import os, subprocess
import pyranges as pr

In [2]:
# Import python package for working with cooler files and tools for analysis
import cooler
import cooltools.lib.plotting

In [3]:
from packaging import version
if version.parse(cooltools.__version__) < version.parse('0.5.4'):
    raise AssertionError("tutorials rely on cooltools version 0.5.4 or higher,"+
                         "please check your cooltools version and update to the latest")

In [4]:
### to load a cooler with a specific resolution use the following syntax:
#### this is the binsize used for eigs
clr_rbp1 = cooler.Cooler("/usr/users/papantonis1/aman/microc_data/nadine_macro/RBP1-MNase-R1-filtered.mcool::resolutions/50000")
clr_ctrl = cooler.Cooler("/usr/users/papantonis1/aman/microc_data/nadine_macro/ctrl-MNase-R1-filtered.mcool::resolutions/50000")

In [5]:
import bioframe
bins = clr_ctrl.bins()[:]
hg38_genome = bioframe.load_fasta('/usr/users/papantonis1/aman/microc_project/refgen/hg38.fa');
## note the next command may require installing pysam
gc_cov = bioframe.frac_gc(bins[['chrom', 'start', 'end']], hg38_genome)
gc_cov.to_csv('hg38_gc_cov_50kb.tsv',index=False,sep='\t')
display(gc_cov)

/usr/users/papantonis1/anaconda3/envs/cool_notebook/lib/python3.10/site-packages/bioframe/extras.py:316: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg = df.groupby("chrom", sort=False)[["start", "end"]].apply(_each)


,chrom,start,end,GC
0,chr1,0,50000,0.484250
1,chr1,50000,100000,0.376740
2,chr1,100000,150000,0.429960
3,chr1,150000,200000,0.486340
4,chr1,200000,250000,0.480564
...,...,...,...,...
61771,chrY,57050000,57100000,0.391180
61772,chrY,57100000,57150000,0.397320
61773,chrY,57150000,57200000,0.469440
61774,chrY,57200000,57227415,0.553775


In [6]:
view_df_ctrl = pd.DataFrame({'chrom': clr_ctrl.chromnames,
                        'start': 0,
                        'end': clr_ctrl.chromsizes.values,
                        'name': clr_ctrl.chromnames}
                      )

view_df_rbp1 = pd.DataFrame({'chrom': clr_rbp1.chromnames,
                        'start': 0,
                        'end': clr_rbp1.chromsizes.values,
                        'name': clr_rbp1.chromnames}
                      )

In [7]:
# obtain first 3 eigenvectors
cis_eigs_ctrl = cooltools.eigs_cis(
                        clr_ctrl,
                        gc_cov,
                        view_df=view_df_ctrl,
                        n_eigs=3,
                        )

cis_eigs_rbp1 = cooltools.eigs_cis(
                        clr_rbp1,
                        gc_cov,
                        view_df=view_df_rbp1,
                        n_eigs=3,
                        )

# cis_eigs[0] returns eigenvalues, here we focus on eigenvectors
eigenvector_track_ctrl = cis_eigs_ctrl[1][['chrom','start','end','E1']]
eigenvector_track_rbp1 = cis_eigs_rbp1[1][['chrom','start','end','E1']]

/usr/users/papantonis1/anaconda3/envs/cool_notebook/lib/python3.10/site-packages/cooltools/lib/checks.py:550: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for name, group in track.groupby(track.columns[0]):
/usr/users/papantonis1/anaconda3/envs/cool_notebook/lib/python3.10/site-packages/cooltools/lib/checks.py:550: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for name, group in track.groupby(track.columns[0]):


In [8]:
df = eigenvector_track_ctrl  # CTRL
for chrom in df['chrom'].unique():
    chrom_df = df[df['chrom'] == chrom]
    chrom_df['chromatin'] = chrom_df['E1'].apply(lambda x: 'A1' if x > 0 else 'B1')
    chrom_df['chromatin'].to_csv(f"{chrom}_ctrl.chromatin", index=False, header=False)

df = eigenvector_track_rbp1  # EED
for chrom in df['chrom'].unique():
    chrom_df = df[df['chrom'] == chrom]
    chrom_df['chromatin'] = chrom_df['E1'].apply(lambda x: 'A1' if x > 0 else 'B1')
    chrom_df['chromatin'].to_csv(f"{chrom}_rbp1.chromatin", index=False, header=False)

/tmp/ipykernel_2129259/451941926.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chrom_df['chromatin'] = chrom_df['E1'].apply(lambda x: 'A1' if x > 0 else 'B1')
/tmp/ipykernel_2129259/451941926.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chrom_df['chromatin'] = chrom_df['E1'].apply(lambda x: 'A1' if x > 0 else 'B1')
/tmp/ipykernel_2129259/451941926.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value ins

In [22]:
# CTRL condition
df = eigenvector_track_ctrl.copy()

for Chromosome in df['Chromosome'].unique():
    Chromosome_df = df[df['Chromosome'] == Chromosome].copy()
    Chromosome_df['type'] = Chromosome_df['E1'].apply(lambda x: 'A1' if x > 0 else 'B1')
    Chromosome_df['val1'] = '*'
    Chromosome_df['val2'] = '*'
    Chromosome_df[['Chromosome', 'Start', 'End', 'type', 'val1', 'val2']].to_csv(
        f"{Chromosome}_ctrl_annotation_openmiChromosome.txt", sep="\t", index=False, header=False
    )

# RBP1 condition
df = eigenvector_track_rbp1.copy()

for Chromosome in df['Chromosome'].unique():
    Chromosome_df = df[df['Chromosome'] == Chromosome].copy()
    Chromosome_df['type'] = Chromosome_df['E1'].apply(lambda x: 'A1' if x > 0 else 'B1')
    Chromosome_df['val1'] = '*'
    Chromosome_df['val2'] = '*'
    Chromosome_df[['Chromosome', 'Start', 'End', 'type', 'val1', 'val2']].to_csv(
        f"{Chromosome}_rbp1_annotation_openmiChromosome.txt", sep="\t", index=False, header=False
    )


In [19]:
df

,Chromosome,Start,End,E1
0,chr1,0,50000,NaN
1,chr1,50000,100000,NaN
2,chr1,100000,150000,NaN
3,chr1,150000,200000,NaN
4,chr1,200000,250000,NaN
...,...,...,...,...
61771,chrY,57050000,57100000,NaN
61772,chrY,57100000,57150000,NaN
61773,chrY,57150000,57200000,NaN
61774,chrY,57200000,57227415,NaN


In [9]:
all_loops = pd.read_csv("/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/jupyter_notes/ml_4_multiclassmodel.csv")
all_loops = all_loops.drop(columns=["Unnamed: 0"])
all_loops

,chr1,start1,end1,chr2,start2,end2,status,ctrl_signal,rbp1_signal,log2FC,...,H3K4me3_signal,rad21_signal,GC_mean,GC_diff,E1_anchor1_ctrl,E1_anchor1_rbp1,E1_anchor2_ctrl,E1_anchor2_rbp1,delta_E1_anchor1,delta_E1_anchor2
0,chr1,1952500,1957500,chr1,2042500,2047500,shared,NaN,NaN,0.000000,...,0.188543,0.024018,0.5773,0.0494,NaN,NaN,NaN,NaN,NaN,NaN
1,chr1,2202500,2207500,chr1,2382500,2387500,shared,NaN,NaN,0.000000,...,0.566125,0.063427,0.6135,0.0578,NaN,NaN,NaN,NaN,NaN,NaN
2,chr1,2412500,2417500,chr1,2552500,2557500,shared,NaN,NaN,0.000000,...,0.188543,0.044699,0.6525,0.0102,NaN,NaN,NaN,NaN,NaN,NaN
3,chr1,3490000,3495000,chr1,3615000,3620000,shared,NaN,NaN,0.000000,...,0.188543,0.024769,0.6002,0.0948,NaN,NaN,NaN,NaN,NaN,NaN
4,chr1,3565000,3570000,chr1,3615000,3620000,shared,NaN,NaN,0.000000,...,0.188543,0.042592,0.5949,0.0842,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32272,chrY,10945000,10950000,chrY,11290000,11295000,gained,0.006078,0.006342,0.061177,...,0.188543,0.026824,0.4545,0.0226,-0.064836,-0.140925,0.270996,0.177009,-0.076090,-0.093987
32273,chrY,10982500,10987500,chrY,11292500,11297500,gained,0.007661,0.005159,-0.570544,...,0.249019,0.130330,0.4609,0.0026,0.120802,-0.100746,0.270996,0.177009,-0.221548,-0.093987
32274,chrY,11292500,11297500,chrY,11722500,11727500,gained,0.001017,0.001021,0.005836,...,0.241160,0.123878,0.4195,0.0854,0.270996,0.177009,0.409095,0.332632,-0.093987,-0.076464
32275,chrY,11530000,11535000,chrY,11760000,11765000,gained,0.004674,0.006621,0.502513,...,0.214433,0.112741,0.3788,0.0120,-1.192141,-1.204536,-1.229772,-1.167283,-0.012394,0.062489


In [10]:
anchor1 = all_loops[['chr1', 'start1', 'end1']].copy()
anchor2 = all_loops[['chr2', 'start2', 'end2']].copy()

In [11]:
anchor1.columns = ['Chromosome', 'Start', 'End']
anchor2.columns = ['Chromosome', 'Start', 'End']
eigenvector_track_ctrl.columns = ['Chromosome', 'Start', 'End', 'E1']
eigenvector_track_rbp1.columns = ['Chromosome', 'Start', 'End', 'E1']

In [12]:
# Create PyRanges
anchor1_gr = pr.PyRanges(anchor1)
anchor2_gr = pr.PyRanges(anchor2)
eig_ctrl_gr = pr.PyRanges(eigenvector_track_ctrl)
eig_rbp1_gr = pr.PyRanges(eigenvector_track_rbp1)

# Intersect
anchor1_ctrl = anchor1_gr.join(eig_ctrl_gr).df[['E1']]
anchor1_rbp1 = anchor1_gr.join(eig_rbp1_gr).df[['E1']]
anchor2_ctrl = anchor2_gr.join(eig_ctrl_gr).df[['E1']]
anchor2_rbp1 = anchor2_gr.join(eig_rbp1_gr).df[['E1']]

In [13]:
all_loops['E1_anchor1_ctrl'] = anchor1_ctrl['E1'].values
all_loops['E1_anchor1_rbp1'] = anchor1_rbp1['E1'].values
all_loops['E1_anchor2_ctrl'] = anchor2_ctrl['E1'].values
all_loops['E1_anchor2_rbp1'] = anchor2_rbp1['E1'].values

#delta
all_loops['delta_E1_anchor1'] = all_loops['E1_anchor1_rbp1'] - all_loops['E1_anchor1_ctrl']
all_loops['delta_E1_anchor2'] = all_loops['E1_anchor2_rbp1'] - all_loops['E1_anchor2_ctrl']


In [14]:
all_loops

,chr1,start1,end1,chr2,start2,end2,status,ctrl_signal,rbp1_signal,log2FC,...,H3K4me3_signal,rad21_signal,GC_mean,GC_diff,E1_anchor1_ctrl,E1_anchor1_rbp1,E1_anchor2_ctrl,E1_anchor2_rbp1,delta_E1_anchor1,delta_E1_anchor2
0,chr1,1952500,1957500,chr1,2042500,2047500,shared,NaN,NaN,0.000000,...,0.188543,0.024018,0.5773,0.0494,NaN,NaN,NaN,NaN,NaN,NaN
1,chr1,2202500,2207500,chr1,2382500,2387500,shared,NaN,NaN,0.000000,...,0.566125,0.063427,0.6135,0.0578,NaN,NaN,NaN,NaN,NaN,NaN
2,chr1,2412500,2417500,chr1,2552500,2557500,shared,NaN,NaN,0.000000,...,0.188543,0.044699,0.6525,0.0102,NaN,NaN,NaN,NaN,NaN,NaN
3,chr1,3490000,3495000,chr1,3615000,3620000,shared,NaN,NaN,0.000000,...,0.188543,0.024769,0.6002,0.0948,NaN,NaN,NaN,NaN,NaN,NaN
4,chr1,3565000,3570000,chr1,3615000,3620000,shared,NaN,NaN,0.000000,...,0.188543,0.042592,0.5949,0.0842,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32272,chrY,10945000,10950000,chrY,11290000,11295000,gained,0.006078,0.006342,0.061177,...,0.188543,0.026824,0.4545,0.0226,-0.064836,-0.140925,0.270996,0.177009,-0.076090,-0.093987
32273,chrY,10982500,10987500,chrY,11292500,11297500,gained,0.007661,0.005159,-0.570544,...,0.249019,0.130330,0.4609,0.0026,0.120802,-0.100746,0.270996,0.177009,-0.221548,-0.093987
32274,chrY,11292500,11297500,chrY,11722500,11727500,gained,0.001017,0.001021,0.005836,...,0.241160,0.123878,0.4195,0.0854,0.270996,0.177009,0.409095,0.332632,-0.093987,-0.076464
32275,chrY,11530000,11535000,chrY,11760000,11765000,gained,0.004674,0.006621,0.502513,...,0.214433,0.112741,0.3788,0.0120,-1.192141,-1.204536,-1.229772,-1.167283,-0.012394,0.062489


In [15]:
# Count rows with any NaNs in the E1 columns
e1_cols = ['E1_anchor1_ctrl', 'E1_anchor1_rbp1', 'E1_anchor2_ctrl', 'E1_anchor2_rbp1']
all_loops[e1_cols].isna().any(axis=1).sum()


740

In [16]:
# Show percentage of missing values for each column
missing_rate = all_loops.isna().mean().sort_values(ascending=False)
print(missing_rate)
high_missing_cols = missing_rate[missing_rate > 0.8]
print(high_missing_cols)

ctrl_signal         0.019457
rbp1_signal         0.017784
delta_E1_anchor2    0.016699
comp_switch_A2      0.016699
E1_anchor2_ctrl     0.016513
comp_switch_A1      0.016327
delta_E1_anchor1    0.016327
E1_anchor1_ctrl     0.015987
E1_anchor2_rbp1     0.014128
E1_anchor1_rbp1     0.013446
H3K4me3_signal      0.002633
end1                0.000000
chr1                0.000000
chr2                0.000000
loop_id             0.000000
distance            0.000000
condition           0.000000
log2FC_clipped      0.000000
log2FC              0.000000
status              0.000000
start2              0.000000
end2                0.000000
start1              0.000000
rad21_signal        0.000000
CTCF_signal         0.000000
source              0.000000
loop_class          0.000000
H3K27me3_signal     0.000000
GC_mean             0.000000
GC_diff             0.000000
dtype: float64
Series([], dtype: float64)


In [17]:
#export
all_loops.to_csv("ml_5_multiclassmodel.csv")